In [8]:
import pandas as pd
from tqdm.notebook import tqdm

# Load the data
current_clerks_proceedings = "/data/krosen5/cache/transformers/Transformer/proceedings_by_clerk_current.xlsx"
clerks = pd.read_excel(current_clerks_proceedings, sheet_name=None)

# Filter for CDC data only
cdc_dictionary = {}
for clerk, df in clerks.items():
    df_for_clerk = df.copy()
    # Filter the dataframe for identifier column where value is 'cdc'
    cdc_df = df_for_clerk[df_for_clerk['identifier'] == 'cdc']
    cdc_dictionary[clerk] = cdc_df

print(f"Initial CDC clerks: {len(cdc_dictionary)}")

# Remove any clerks from the dictionary where the dataframe is empty
cdc_dictionary = {k: v for k, v in cdc_dictionary.items() if not v.empty}
print(f"CDC clerks with data: {len(cdc_dictionary)}")

# Filter for MAG data only
mag_dictionary = {}
for clerk, df in clerks.items():
    df_for_clerk = df.copy()
    # Filter the dataframe for identifier column where value is 'mag'
    mag_df = df_for_clerk[df_for_clerk['identifier'] == 'mag']
    mag_dictionary[clerk] = mag_df

print(f"Initial MAG clerks: {len(mag_dictionary)}")

# Remove any clerks from the dictionary where the dataframe is empty
mag_dictionary = {k: v for k, v in mag_dictionary.items() if not v.empty}
print(f"MAG clerks with data: {len(mag_dictionary)}")

Initial CDC clerks: 158
CDC clerks with data: 119
Initial MAG clerks: 158
MAG clerks with data: 78


In [37]:
mag_dictionary_3 = mag_dictionary.copy()
cdc_dictionary_3 = cdc_dictionary.copy()

In [39]:
from regex_labels.patterns_and_extractors import (
determine_bond_status
)
# filter out bond set and bond not set cases
def filter_court_data(court_dictionary):
    for clerk, df in court_dictionary.items():
        filtered_df = df[~df['text'].apply(lambda x: determine_bond_status(x) in ['BOND_SET', 'BOND_NOT_SET'])]
        # Update the dictionary with the filtered dataframe
        filtered_df = filtered_df.reset_index(drop=True)
        court_dictionary[clerk] = filtered_df
    return court_dictionary

mag_dictionary_3 = filter_court_data(mag_dictionary_3)
cdc_dictionary_3 = filter_court_data(cdc_dictionary_3)


In [60]:
#make a dataset for mag and clerk with filtered data where you combine all the dataframes into one dataframe
mag_filtered = pd.concat(mag_dictionary_3.values(), ignore_index=True)
cdc_filtered = pd.concat(cdc_dictionary_3.values(), ignore_index=True)

#then we filter for rows that have any of the following
following_words = [
    r'\bBOND\s*SET\b', 
    r'\bSET\s*BOND\b', 
    r'\bPRE(?:-)?SET\s*BOND\b', 
    r'\bBOND\s*PRE(?:-)?SET\b', 
    r'\bBAIL\s*ORDER\b'
]
pattern = '|'.join(following_words)
mag_filtered_final = mag_filtered[mag_filtered['text'].str.contains(pattern, case=False, na=False, regex=True)]
cdc_filtered_final = cdc_filtered[cdc_filtered['text'].str.contains(pattern, case=False, na=False, regex=True)]

mag_filtered_final = mag_filtered_final.reset_index(drop=True)
cdc_filtered_final = cdc_filtered_final.reset_index(drop=True)

#only chosoe the text column
mag_filtered_final = mag_filtered_final[['text']]
cdc_filtered_final = cdc_filtered_final[['text']]

#drop duplicates
mag_filtered_final = mag_filtered_final.drop_duplicates().reset_index(drop=True)
cdc_filtered_final = cdc_filtered_final.drop_duplicates().reset_index(drop=True)

In [61]:
import re
combo_pattern = re.compile(r'(?:' + '|'.join(following_words) + r')', re.IGNORECASE)
def merge_multiline_text(text):
    """Split multiline text, merge adjacent lines if they form target phrases,
       trim whitespace, and drop duplicates within the same row."""
    cells = [line.strip() for line in str(text).splitlines() if line.strip()]
    out = []
    i = 0
    while i < len(cells):
        cur = cells[i]
        if i + 1 < len(cells):
            nxt = cells[i+1]
            combined = re.sub(r'\s+', ' ', f"{cur} {nxt}".strip())
            if combo_pattern.search(combined):
                cur = combined
                i += 1  # skip the next since we merged
        # add if not a duplicate of the last one we kept
        if not out or out[-1] != cur:
            out.append(cur)
        i += 1
    return out

def process_df(df):
    """Process the dataframe to merge multiline text, trim whitespace,
       and deduplicate rows."""
    # Apply merging + trimming + row-level deduplication
    df["merged_cells"] = df["text"].apply(merge_multiline_text)

    # Explode into one row per merged cell, then drop duplicates across entire DF
    lines_df = (
        df.explode("merged_cells", ignore_index=True)
          .rename(columns={"merged_cells": "line"})[["line"]]
          .drop_duplicates(ignore_index=True)
    )
    return lines_df

mag_filtered_final_2 = process_df(mag_filtered_final)
cdc_filtered_final_2 = process_df(cdc_filtered_final)

#now only keep the rows that have any of the following words
mag_filtered_final_2 = mag_filtered_final_2[mag_filtered_final_2['line'].str.contains(pattern, case=False, na=False, regex=True)].reset_index(drop=True)
cdc_filtered_final_2 = cdc_filtered_final_2[cdc_filtered_final_2['line'].str.contains(pattern, case=False, na=False, regex=True)].reset_index(drop=True)

In [62]:
mag_output_path_for_filtered = "/data/krosen5/cache/transformers/Transformer/proceedings_by_clerk_current_mag_filtered_v2.xlsx"
#make the mag filtered final into an excel file where it is only one sheet now since it is all combined
with pd.ExcelWriter(mag_output_path_for_filtered) as writer:
    mag_filtered_final_2.to_excel(writer, sheet_name="MAG_Filtered", index=False)  # Sheet names must be <= 31 characters
cdc_output_path_for_filtered = "/data/krosen5/cache/transformers/Transformer/proceedings_by_clerk_current_cdc_filtered_v2.xlsx"
#make the cdc filtered final into an excel file where it is only one sheet now since it is all combined
with pd.ExcelWriter(cdc_output_path_for_filtered) as writer:
    cdc_filtered_final_2.to_excel(writer, sheet_name="CDC_Filtered", index=False)  # Sheet names must be <= 31 characters

In [40]:
mag_output_path_for_filtered = "/data/krosen5/cache/transformers/Transformer/proceedings_by_clerk_current_mag_filtered.xlsx"
with pd.ExcelWriter(mag_output_path_for_filtered) as writer:
    for clerk, df in mag_dictionary_3.items():
        df.to_excel(writer, sheet_name=clerk[:31], index=False)  # Sheet names must be <= 31 characters
cdc_output_path_for_filtered = "/data/krosen5/cache/transformers/Transformer/proceedings_by_clerk_current_cdc_filtered.xlsx"
with pd.ExcelWriter(cdc_output_path_for_filtered) as writer:
    for clerk, df in cdc_dictionary_3.items():
        df.to_excel(writer, sheet_name=clerk[:31], index=False)  # Sheet names must be <= 31 characters

In [29]:
from regex_labels.patterns_and_extractors import (
compiled_not_court_pattern
)

second_part_of_statement = r"""
AND\s*(?:IS\s*|WAS\s*)?
(?:
    NOT\s*(?:BROUGHT|TRANSPORTED|TRANSFERRED)
    |
    (?:REFUSED\s*|UNABLE\s*|FAILED\s*)
    (?:
        TO\s*(?:BE\s*(?:BROUGHT|TRANSPORTED)|APPEAR|COME|BROUGHT|LEAVE)
        |
        TRANSPORTED
    )
    |
    (?:NOT\s*(?:BEING\s*)?(?:PRESENT|AVAILABLE|BROUGHT(?:\s*OVER)?|TRANSPORTED(?:\s*OVER)?)|UNAVAILABLE|ABSENT)
    |
    DID\s*NOT\s*(?:APPEAR|COME|MAKE\s*AN\s*APPEARANCE)
)
(?:\s*(?:(?:FOR\s*|AT\s*|ON\s*)?(?:THIS\s*|THE\s*)?(?:PROCEE?DING|HEARING|SETTING|DAY|MATTER)|DAY|IN\s*COURT(?:\s*TODAY)?|TODAY|TODAY'S\s*COURT\s*SETTING|TO\s*COME))?
(?:\s*
    (?:(?:TO|INTO|IN\s*TO|UP(?:\s*FOR|\s*IN|\s*TO)?)\s*)?
    (?:
        (?:ZOOM\s*OR\s*)?OPEN\s*COURT
        |
        OPEN/ZOOM\s*COURT
        |
        ZOOM\s*(?:COURT|HEARING|MEETING)
        |
        COURT
        |
        TO\s*VIRTUAL\s*COURTROOM
    )
)?
(?:\s*(?:(?:FOR\s*|AT\s*|ON\s*)?(?:THIS\s*|THE\s*)?(?:PROCEE?DING|HEARING|SETTING|DAY|MATTER)|DAY|IN\s*COURT(?:\s*TODAY)?|TODAY|TODAY'S\s*COURT\s*SETTING|TO\s*COME))?
"""

from tqdm.notebook import tqdm
tqdm.pandas()

#compile the second_part of the statement regex
import re
compiled_second_part_of_statement = re.compile(second_part_of_statement, re.VERBOSE |re.IGNORECASE|re.MULTILINE)

In [34]:
cdc_dictionary_2 = cdc_dictionary.copy()
mag_dictionary_2 = mag_dictionary.copy()

In [35]:
from tqdm.notebook import tqdm
tqdm.pandas()

def filter_court_data(dictionary):
    """Filter court data based on regex patterns and extract match values"""
    for clerk, df in tqdm(dictionary.items()):
        df_for_clerk = df.copy()
        
        # Apply the compiled regex and extract the match value (or None if no match)
        df_for_clerk['not_court'] = df_for_clerk['text'].apply(
            lambda x: match.group() if (match := compiled_not_court_pattern.search(x)) else pd.NA
        )
        df_for_clerk['second_part_of_statement'] = df_for_clerk['text'].apply(
            lambda x: match.group() if (match := compiled_second_part_of_statement.search(x)) else pd.NA
        )
        
        # Filter rows where either column has a non-None value
        filtered_df = df_for_clerk[(df_for_clerk['not_court'].notna()) | (df_for_clerk['second_part_of_statement'].notna())]
        dictionary[clerk] = filtered_df
    return dictionary


# Apply the function to both dictionaries
cdc_dictionary_2 = filter_court_data(cdc_dictionary_2)
mag_dictionary_2 = filter_court_data(mag_dictionary_2)

  0%|          | 0/119 [00:00<?, ?it/s]

  0%|          | 0/78 [00:00<?, ?it/s]

,case_var,date,clerk,text,identifier,not_court,second_part_of_statement


In [36]:
mag_output_path = "/data/krosen5/cache/transformers/Transformer/proceedings_by_clerk_mag_not_court_or_second_part.xlsx"
with pd.ExcelWriter(mag_output_path) as writer:
    for clerk, df in mag_dictionary_2.items():
        if not df.empty:
            df.to_excel(writer, sheet_name=clerk[:31], index=False)  # Sheet names can be max 31 characters

cdc_output_path = "/data/krosen5/cache/transformers/Transformer/proceedings_by_clerk_cdc_not_court_or_second_part.xlsx"
with pd.ExcelWriter(cdc_output_path) as writer:
    for clerk, df in cdc_dictionary_2.items():
        if not df.empty:
            df.to_excel(writer, sheet_name=clerk[:31], index=False)  # Sheet names can be max 31 characters